# Notebook 08 — IRASA fitting: iEEG (Frauscher 2018)

Decomposes iEEG spectra into aperiodic and periodic components using **IRASA**
(Irregular Resampling Auto-Spectral Analysis; Wen & Liu, 2016).  
Uses identical data loading and Welch settings (2 s segments, 1 s overlap) as NB 02,
enabling direct method comparison in NB 09.

**Frequency range**: 2–40 Hz.  IRASA requires `band[1] * h_max < fs/2` (Nyquist).
With `h_max = 2.0` and `fs = 200 Hz`, the upper band must be < 50 Hz.
Note: specparam (NB 02) was fit on 2–100 Hz; the comparison in NB 09 is therefore
restricted to parameters derived from the same 2–40 Hz range.

**Runtime note**: IRASA resamples at 20 different h-factors — expect 5–15 min.

**Peak memory**: ~3.8 GB (20 h-factors × 1772 × 13600 × float64).

**Outputs** saved to `data/interim/`:
- `irasa_ieeg_results.csv` — long-format (one row per detected peak; NaN if no peaks)
- `irasa_ieeg_ch_summary.csv` — per-channel summary (one row per channel)
- `region_aperiodic_irasa_ieeg.csv` — region-level median exponent/offset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io

from pesco.irasa import (
    inspect_irasa_fit_quality,
    inspect_irasa_fits,
    irasa2pandas,
    run_irasa,
)

In [ ]:
PROJECT_ROOT = Path("../../").resolve()
print(f"Project root: {PROJECT_ROOT}")

DATA_DIR     = PROJECT_ROOT / "data"
IEEG_MAT     = DATA_DIR / "Frauscher2018" / "WakefulnessMatlabFile.mat"
CHANNEL_INFO = DATA_DIR / "Frauscher2018" / "ChannelInformation.csv"
REGION_INFO  = DATA_DIR / "Frauscher2018" / "RegionInformation.csv"
INTERIM_DIR  = DATA_DIR / "interim"
INTERIM_DIR.mkdir(exist_ok=True)

FS           = 200      # Hz
R2_THRESHOLD = 0.80

# IRASA constraint: band[1] * h_max < fs/2 (Nyquist = 100 Hz).
# With h_max=2.0: band[1] must be < 50 Hz.
# Using 40 Hz is safe (40 * 2.0 = 80 Hz << 100 Hz) and matches
# the robustness range tested in NB 07.
IRASA_SETTINGS = dict(
    band=(2.0, 40.0),
    nperseg=400,          # 2 s @ 200 Hz
    noverlap=200,         # 1 s overlap
    hset_info=(1.05, 2.0, 0.05),
)
PEAK_SETTINGS = dict(
    peak_threshold=2.5,
    min_peak_height=0.0,
    peak_width_limits=(0.5, 12.0),
    smoothing_window=1,
)

## 1. Load iEEG data

Identical loading logic as NB 02.

In [4]:
print(f"Loading {IEEG_MAT.name} …")

try:
    mat = scipy.io.loadmat(str(IEEG_MAT))
    data       = mat["Data"].T
    Fs         = float(np.array(mat["SamplingFrequency"]).flatten()[0])
    ch_regions = np.array(mat["ChannelRegion"]).flatten()
    ch_names   = [str(x[0][0]) for x in mat["ChannelName"]]
    print("Loaded via scipy.io.loadmat")
except Exception as e:
    print(f"scipy.io failed ({e}), trying h5py …")
    with h5py.File(IEEG_MAT, "r") as f:
        data         = np.array(f["Data"]).T
        Fs           = float(np.array(f["SamplingFrequency"]).flatten()[0])
        ch_regions   = np.array(f["ChannelRegion"]).flatten()
        ch_names_refs = np.array(f["ChannelName"]).flatten()
        ch_names = [
            "".join(chr(c) for c in np.array(f[ref]).flatten())
            for ref in ch_names_refs
        ]
    print("Loaded via h5py")

print(f"Data shape : {data.shape}  |  Fs: {Fs} Hz")
print(f"Duration   : {data.shape[1] / Fs:.1f} s  |  Channels: {len(ch_names)}")

Loading WakefulnessMatlabFile.mat …
Loaded via scipy.io.loadmat
Data shape : (1772, 13600)  |  Fs: 200.0 Hz
Duration   : 68.0 s  |  Channels: 1772


## 2. Run IRASA

IRASA separates each spectrum into a fractal (aperiodic) component and an
oscillatory (periodic) component by exploiting the fact that resampling
a signal by a factor h shifts peaks in frequency but leaves the 1/f background
unchanged.  The geometric mean of the up- and down-resampled spectra provides
a peak-free estimate of the aperiodic component.

In [5]:
print(f"Running IRASA on {data.shape[0]} channels …")
print(f"Settings: {IRASA_SETTINGS}")
print("(This may take 5–15 minutes.)")

irasa_result = run_irasa(data, fs=FS, **IRASA_SETTINGS)

print(f"\nFrequencies   : {irasa_result.freqs[0]:.2f} – {irasa_result.freqs[-1]:.2f} Hz  "
      f"({len(irasa_result.freqs)} bins)")
print(f"raw_spectrum  : {irasa_result.raw_spectrum.shape}")
print(f"aperiodic     : {irasa_result.aperiodic.shape}")
print(f"periodic      : {irasa_result.periodic.shape}")

Running IRASA on 1772 channels …
Settings: {'band': (2.0, 100.0), 'nperseg': 400, 'noverlap': 200, 'hset_info': (1.05, 2.0, 0.05)}
(This may take 5–15 minutes.)


AssertionError: The evaluated frequency range goes up to 200.0Hz which is higher than the Nyquist frequency for your data of 100.0Hz, 
try to either lower the upper bound for the hset or decrease the upper band limit, when running IRASA.

Quick sanity check — median aperiodic and periodic spectra.

In [ ]:
freqs = irasa_result.freqs

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(freqs, np.median(irasa_result.raw_spectrum, axis=0),
            label="raw", color="black", linewidth=1)
ax.semilogy(freqs, np.median(irasa_result.aperiodic, axis=0),
            label="aperiodic", color="tomato", linewidth=1.5)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (V²/Hz)")
ax.set_title("Median spectrum — raw vs aperiodic")
ax.legend()

ax = axes[1]
ax.plot(freqs, np.median(irasa_result.periodic, axis=0),
        color="steelblue", linewidth=1)
ax.axhline(0, color="grey", linewidth=0.5, linestyle=":")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Periodic power (V²/Hz)")
ax.set_title("Median periodic (oscillatory) component")

fig.tight_layout()
plt.show()

## 3. Extract aperiodic parameters and peaks

In [ ]:
results_df = irasa2pandas(irasa_result, fit_func="fixed", **PEAK_SETTINGS)
print(f"Shape: {results_df.shape}")
results_df.head()

## 4. QC: inspect fit quality

In [ ]:
ch_df, summary_df, fig, axes = inspect_irasa_fit_quality(
    results_df, r2_threshold=R2_THRESHOLD, bins=40, show=True
)

In [ ]:
fig, axes = inspect_irasa_fits(
    irasa_result, results_df=results_df, select_by="exponent", n_spectra=5
)

## 5. Attach channel and region metadata

Identical join logic as NB 02.

In [ ]:
# Channel metadata
ch_info = pd.read_csv(CHANNEL_INFO)
ch_info.columns = ch_info.columns.str.strip()
for col in ch_info.select_dtypes("object").columns:
    ch_info[col] = ch_info[col].str.strip("'")

# Region labels from the MATLAB file
ch_region_df = pd.DataFrame({
    "ch_name": ch_names,
    "Region":  ch_regions.astype(int),
})

# Region names
region_info = pd.read_csv(REGION_INFO)
region_info.columns = region_info.columns.str.strip()
for col in region_info.select_dtypes("object").columns:
    region_info[col] = region_info[col].str.strip("'")

ch_meta = ch_region_df.merge(region_info, on="Region", how="left")
ch_meta = ch_meta.merge(
    ch_info.rename(columns={"Channel name": "ch_name"}),
    on="ch_name", how="left"
)
ch_meta["ID"] = range(len(ch_meta))

results_full = results_df.merge(ch_meta, on="ID", how="left")
results_full["good_fit"] = results_full["gof_rsquared"] >= R2_THRESHOLD

print(f"Shape: {results_full.shape}")
results_full.head(3)

## 6. Regional summary

In [ ]:
ch_summary = results_full.drop_duplicates(subset="ID").drop(
    columns=["cf", "pw", "bw"], errors="ignore"
)

if "Lobe" in ch_summary.columns:
    print("Good fits by lobe:")
    print(
        ch_summary[ch_summary["good_fit"]]
        .groupby("Lobe")["exponent"]
        .agg(["count", "median"])
        .round(3)
    )

In [ ]:
def region_summary_irasa(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate channel-level IRASA results to region level (median)."""
    good = df[df["good_fit"]].copy()
    good = good.dropna(subset=["Region name", "exponent", "offset"])
    grp = good.groupby(["Region name", "Lobe"], as_index=False)
    return grp.agg(
        exponent_median=("exponent", "median"),
        exponent_iqr=("exponent", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        exponent_n=("exponent", "count"),
        offset_median=("offset", "median"),
        offset_iqr=("offset", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    )


irasa_region = region_summary_irasa(ch_summary)
print(irasa_region.sort_values("exponent_median", ascending=False).to_string(index=False))

## 7. Save results

In [ ]:
# Long-format (one row per peak)
out_path = INTERIM_DIR / "irasa_ieeg_results.csv"
results_full.to_csv(out_path, index=False)
print(f"Saved {len(results_full)} rows → {out_path}")

# Per-channel summary
ch_summary.to_csv(INTERIM_DIR / "irasa_ieeg_ch_summary.csv", index=False)
print(f"Saved channel summary ({len(ch_summary)} rows) → "
      f"{INTERIM_DIR / 'irasa_ieeg_ch_summary.csv'}")

# Region-level medians
irasa_region.to_csv(INTERIM_DIR / "region_aperiodic_irasa_ieeg.csv", index=False)
print(f"Saved region summary ({len(irasa_region)} regions) → "
      f"{INTERIM_DIR / 'region_aperiodic_irasa_ieeg.csv'}")